# 第5回 正則化による多重共線性の問題への対処（解答例・教員用）


---


## 前回からの接続


第4回は、住宅の査定担当者として、価格を当てる式を交差検証（cross-validation）で選びました。車庫の台数と車庫の面積のような似た2列があると係数が符号まで揺れること（多重共線性）を見て、似た2列のうち台数のほうを外し、10本の列の式で査定しました。交差検証の平均絶対誤差（MAE）は19,297ドルでした。

ただ、データには80本もの列があります。列を増やせば、もっと当たるかもしれません。そのかわり、似た列も増えていきます。1本ずつ手で外していくのは、もう無理です。今日は、列を増やしても係数が落ち着くよう、係数そのものに罰をかける方法を使います。


---


## 今日の分析目標


あなたは前回と同じ、住宅の査定担当者です。列を増やして当たりを良くしながら、お客さんに説明できる係数を保てるかに答える。

手がかりの数の列を30本に増やして、交差検証で当たりを測り、係数がどうなるかを確かめます。そのうえで、係数の大きさに罰をかける正則化（リッジとラッソ）で係数を落ち着かせ、罰の強さを交差検証で選びます。

もう1つのねらいは、初めての道具でも、ドキュメント（説明書き）で引数を読んでから使えるようになることです。今日は、リッジ・ラッソ・`cross_val_score`という初めての道具を、ドキュメントを読んでから使います。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、前回と同じ、米国エイムズ市の住宅の査定担当者です。家の条件から値段を見積もり、お客さんに、その値段になった理由を条件ごとに説明します。

前回は、手がかりを10本の列に絞りました。今日は、もっと多くの条件を手がかりにして、見積もりの外れを小さくしたいと考えています。


### 手元のデータ

前回と同じ、エイムズ市の資産評価事務所が記録した、2006〜2010年の住宅の売買2,930件です（De Cock, 2011）。1行が1軒の売買で、答えの列は売れた価格（米ドル）です。前回と同じく、ふつうの売買で、居住面積が4,000平方フィート以下の2,412軒を使います。

今日は、数の列を30本使います（0節）。


### 最後に出す答え

30本の列で当たりを良くした、査定に使う式と、その式の係数で言える説明です。そして、その式による1軒の家の査定額です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 足し合わせるとほかの列になる列があると、係数が読めなくなる理由を説明できる（完全な多重共線性）
- 正則化（regularization）が、係数の大きさに罰をかけて係数を落ち着かせることを、目的関数の式で説明できる
- 罰をかける前に標準化（standardization）する理由を説明し、交差検証の各回で訓練だけを見て標準化できる
- リッジ（Ridge）は係数を縮めて似た列で分け合い、ラッソ（Lasso）はいくつかの係数をちょうど0にすることを確かめられる
- 罰の強さ`alpha`を、交差検証で選べる
- ドキュメントで、引数の意味と目的関数の式を読んでから、初めての道具を使える
- モデルを関数の引数として渡し、交差検証の関数の「式を作る」部品を差し替えられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

今日は、正則化の道具`Ridge`と`Lasso`、交差検証の道具`cross_val_score`が加わります。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import KFold, cross_val_score


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### お手本：表を読み込んで、列名を日本語にする

- 役割：表を読み込み、今日使う33本の列の名前を日本語に付け替える
- 入力：データの置き場所`DATA_DIR`と、ファイル名`ames_housing.csv`
- 出力：表`raw`と、その件数

第4回と同じ`rename`です。今日は、売買の条件・価格・石張りの種類と、手がかりにする数の列30本の名前を付け替えます。中身は、セルの下の表で確かめてください。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/ames_housing.csv')
ja_names = {
    'Sale Condition': '売買の条件', 'SalePrice': '価格',
    'Mas Vnr Type': '石張りの種類', 'Lot Area': '敷地面積',
    'Overall Qual': '品質', 'Overall Cond': '状態',
    'Year Built': '建てた年', 'Year Remod/Add': '改装した年',
    'Mas Vnr Area': '石張りの面積', 'BsmtFin SF 1': '地下の仕上げ1',
    'BsmtFin SF 2': '地下の仕上げ2', 'Bsmt Unf SF': '地下の未仕上げ',
    'Total Bsmt SF': '地下の面積', '1st Flr SF': '1階の面積',
    '2nd Flr SF': '2階の面積', 'Low Qual Fin SF': '低品質の面積',
    'Gr Liv Area': '居住面積', 'Bsmt Full Bath': '地下のバスルーム',
    'Bsmt Half Bath': '地下の洗面所', 'Full Bath': 'バスルーム数',
    'Half Bath': '洗面所の数', 'Bedroom AbvGr': '寝室数',
    'Kitchen AbvGr': 'キッチン数', 'TotRms AbvGrd': '部屋数',
    'Fireplaces': '暖炉の数', 'Garage Cars': '車庫の台数',
    'Garage Area': '車庫の面積', 'Wood Deck SF': 'ウッドデッキ',
    'Open Porch SF': '開放ポーチ', 'Enclosed Porch': '囲いポーチ',
    '3Ssn Porch': '三季ポーチ', 'Screen Porch': '網戸ポーチ',
    'Pool Area': 'プールの面積'}
raw = raw.rename(columns=ja_names)
print(len(raw), '件')


第4回で使った列のほかに、新しく加わった列の中身は次のとおりです。

| 日本語の名前 | 元の名前 | 中身 |
|---|---|---|
| `状態` | `Overall Cond` | 家全体の状態（1〜10の10段階） |
| `石張りの種類`・`石張りの面積` | `Mas Vnr Type`・`Mas Vnr Area` | 外壁の石やれんがの張り方と、その面積 |
| `地下の仕上げ1`・`地下の仕上げ2`・`地下の未仕上げ` | `BsmtFin SF 1`ほか | 地下室のうち、仕上げた部分（2種類）と仕上げていない部分の面積 |
| `1階の面積`・`2階の面積`・`低品質の面積` | `1st Flr SF`ほか | 1階・2階の面積と、仕上げの質が低い部分の面積 |
| `地下のバスルーム`・`地下の洗面所` | `Bsmt Full Bath`ほか | 地下にあるバスルームと洗面所（シャワーのないもの）の数 |
| `洗面所の数`・`寝室数`・`キッチン数` | `Half Bath`ほか | 地上階の洗面所・寝室・キッチンの数 |
| `ウッドデッキ`・`開放ポーチ`・`囲いポーチ`・`三季ポーチ`・`網戸ポーチ` | `Wood Deck SF`ほか | 家の外の、デッキや屋根付きの縁側（ポーチ）の面積 |
| `プールの面積` | `Pool Area` | プールの面積 |

数の列のうち、使わない列もあります。区分を番号で表した列（`MS SubClass`。第3回で見たように、番号の大小に意味がない）、売れた月と年（家の条件ではない）、その他の設備の金額、そして欠測（missing data）の理由が分からない間口の長さ（`Lot Frontage`）と、車庫の無い家で空欄になる車庫を建てた年（`Garage Yr Blt`）です。


### 変えてみる：ふつうの売買に絞り、面積を㎡に直す

- 役割：第4回と同じく2,412軒に絞り、面積の17本の列を㎡に直す
- 入力：表`raw`
- 出力：表`df`の軒数

第4回のセルと同じ形です。面積の列の名前を`area`にしまってから、`for`文で0.0929を掛けます。


In [ ]:
normal = raw['売買の条件'] == 'Normal'     # ふつうの売買
small = raw['居住面積'] <= 4000            # 広すぎない家
df = raw[normal & small].reset_index(drop=True)
area = ['敷地面積', '石張りの面積', '地下の仕上げ1', '地下の仕上げ2',
        '地下の未仕上げ', '地下の面積', '1階の面積', '2階の面積',
        '低品質の面積', '居住面積', '車庫の面積', 'ウッドデッキ',
        '開放ポーチ', '囲いポーチ', '三季ポーチ', '網戸ポーチ',
        'プールの面積']
for c in area:
    df[c] = df[c] * 0.0929                 # 平方フィート → ㎡
print(len(df), '軒')


### お手本：欠測した理由を確かめる

- 役割：手がかりにする30本の列の欠測を数え、欠測した家の様子を見る
- 入力：表`df`と、30本の列の名前`cols`
- 出力：欠測のある列と件数、石張りの種類が空欄の家の面積、地下のバスルームが欠測した家の地下の面積

第3回で学んだとおり、欠測を見つけたら、すぐに1つの値で補完（imputation）せず、まず理由を確かめます。


In [ ]:
cols = ['敷地面積', '品質', '状態', '建てた年', '改装した年',
        '石張りの面積', '地下の仕上げ1', '地下の仕上げ2',
        '地下の未仕上げ', '地下の面積', '1階の面積', '2階の面積',
        '低品質の面積', '居住面積', '地下のバスルーム',
        '地下の洗面所', 'バスルーム数', '洗面所の数', '寝室数',
        'キッチン数', '部屋数', '暖炉の数', '車庫の台数',
        '車庫の面積', 'ウッドデッキ',
        '開放ポーチ', '囲いポーチ', '三季ポーチ', '網戸ポーチ',
        'プールの面積']
print(len(cols), '本')
print(df[cols].isna().sum()[df[cols].isna().sum() > 0])
no_type = df[df['石張りの種類'].isna()]          # 種類が空欄の家
known = no_type['石張りの面積'].notna()   # 面積が分かる家
print((no_type['石張りの面積'] == 0).sum(), known.sum())
print(df[df['地下のバスルーム'].isna()]['地下の面積'])


欠測しているのは3つの列で、石張りの面積が11件、地下のバスルームと地下の洗面所が1件ずつです。

石張りの面積が欠測した11件は、どれも石張りの種類も空欄です。種類が空欄で面積の分かる家は1,481軒あり、そのうち1,474軒は面積が0でした。種類が空欄の家は、ほとんどが石張りの無い家です。欠測した11件も、石張りの無い家として面積を0に戻します。

地下のバスルームが欠測した家は、地下の面積が0、つまり地下室の無い家でした。地下室が無ければ、地下のバスルームも洗面所も0です。

どちらも、ほかの列から真の値（0）が分かる欠測です。平均で埋めるのではなく、0に戻します。


In [ ]:
fill0 = ['石張りの面積', '地下のバスルーム', '地下の洗面所']
df[fill0] = df[fill0].fillna(0)
print(df[cols].isna().sum().sum(), '件')


30本の列の欠測は0件になりました。


### 補完するか、使わないか

線形回帰（linear regression）も、このあと使うリッジやラッソも、手がかりの表に欠測（`NaN`）が1つでもあると、第3回で見た`ValueError: Input X contains NaN.`で止まります。欠測を残したまま式は作れないので、使う列の欠測は、式を作る前に必ずなくします。

なくし方は3つあります。

1. 真の値がほかの列から分かるなら、戻す（第3回のバスルームの表記、今日の石張りの面積と地下のバスルーム）
2. ほかの列から予測して補完する（第3回の寝室数とベッド数の回帰補完）
3. その列（または行）を、使わない

第3回では2を中心に学びました。でも、いつも補完すればよいわけではありません。補完した値は、どこまでいっても推測です。次のような場合は、3の「使わない」を選ぶほうがふつうです。

- 値がそもそも存在しない：車庫の無い家の「車庫を建てた年」は、欠けているのではなく、存在しません（第3回の、レビューが0件の部屋の点数と同じ）
- 欠けた理由が分からず、欠けが多い：補完した値が当たっているかを確かめる手立てがない
- 測った値そのものが信頼できない：遺伝子発現データでは、発現量が測定の下限に近い遺伝子（低発現遺伝子）は、値が欠けたり、雑音ばかりだったりします。こうした遺伝子は、補完せずに、解析の前に外すのがふつうです。発展で使うラットの目のデータも、元の論文では、目の組織で発現していると判定された遺伝子だけを解析しています

今日の間口の長さは、2つ目にあたります。


In [ ]:
frontage = raw.loc[normal & small, 'Lot Frontage']
frontage = frontage.reset_index(drop=True)   # df と行番号をそろえる
print(frontage.isna().sum(), round(frontage.isna().mean() * 100, 1))
print(df.groupby(frontage.isna())[['敷地面積', '価格']].median())


間口の長さは、451件（18.7%）が欠けています。欠けた家（True）は、欠けていない家（False）より、敷地が広く、価格も高めです。欠けるかどうかが価格と関係しているので、でたらめな欠け方ではありません。どんな理由で欠けたのかは、データからは分かりません。

補完の良し悪しを確かめるには、第3回のように答えの分かる欠測を作って評価する必要があります。今日はその手間をかけず、間口の長さは使わないことにします。使わないと決めたことと、その理由を書き残しておくのも、前処理の大事な仕事です。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、②の解析（正則化）と、③の解釈が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の5つの小さな問いに答えていきます。

1. 列を増やすと？（1節）
2. 係数に罰をかけると？（2節）
3. 係数をちょうど0にできる？（3節）
4. 罰の強さは、どう選ぶ？（4節）
5. 説明できる係数で、いくらと査定する？（5節）


---


## 1. 列を増やすと？


30本の列がそろいました。まず、第4回と同じ交差検証で、列を増やした式の当たりを測ります。


### おさらい：第4回の交差検証の関数

第4回で作った`mae`と`cv_score`を、そのまま写します。`cv_score`は、5分割の交差検証で、毎回のMAEと、`watch`で指定した列の係数をためた表を返す関数でした。今日は係数を`watch`で必ず指定するので、既定値は外し、RMSEも省きます。

- 役割：5つの組を作り、交差検証で式を評価する関数を用意する
- 入力：手がかりの列の名前`cols`と、係数を見る列`watch`
- 出力：5行の表（列はMAEと係数）


In [ ]:
def mae(y, pred):          # MAE
    return round(np.abs(y - pred).mean())

kf = KFold(n_splits=5, shuffle=True, random_state=0)

def cv_score(cols, watch):
    rows = []
    for tr_i, te_i in kf.split(df):
        tr_s, te_s = df.iloc[tr_i], df.iloc[te_i]
        m_s = LinearRegression().fit(tr_s[cols], tr_s['価格'])
        coef_s = pd.Series(m_s.coef_, index=cols)
        pred_s = m_s.predict(te_s[cols])
        rows.append({'MAE': mae(te_s['価格'], pred_s),
                     '係数': round(coef_s[watch])})
    return pd.DataFrame(rows)


### お手本：10本と30本を比べる

- 役割：第4回の10本の列の式と、30本の列の式を、同じ交差検証で評価する
- 入力：10本の列の名前`feat10`と、30本の列の名前`cols`
- 出力：2つの式の5回分のMAEと、居住面積の係数


In [ ]:
feat10 = ['品質', '居住面積', '地下の面積', '車庫の面積',
          '建てた年', '改装した年', 'バスルーム数', '部屋数',
          '暖炉の数', '敷地面積']
cv10 = cv_score(feat10, '居住面積')
cv30 = cv_score(cols, '居住面積')
print(pd.DataFrame({'10本のMAE': cv10['MAE'],
                    '30本のMAE': cv30['MAE'],
                    '10本の係数': cv10['係数'],
                    '30本の係数': cv30['係数']}))
print(round(cv10['MAE'].mean()), round(cv30['MAE'].mean()))


10本の式のMAEの平均は19,297ドルで、第4回と同じです。30本にすると17,188ドルになり、2千ドル以上小さくなりました。5回のどの組でも、30本の式のほうが当たります。列を増やすと、当たりは良くなるのです。

ところが、居住面積の係数を見てください。10本の式では1㎡あたり675〜705ドルでしたが、30本の式では358〜462ドルです。小さくなったうえに、回ごとの揺れも大きくなっています。


### TODO①：居住面積は、何の和？

新しく加わった列に、`1階の面積`・`2階の面積`・`低品質の面積`があります。居住面積は、この3つの列の和と同じではないでしょうか。2,412軒それぞれで、居住面積から3つの列の和を引いた差を計算し、差の大きさのいちばん大きな値を表示してください。


In [ ]:
# TODO: 居住面積から3つの列の和を引いた差の、大きさの最大値を表示する

# 解答例①：居住面積と、3つの列の和の差
diff = df['居住面積'] - (df['1階の面積'] + df['2階の面積']
                       + df['低品質の面積'])
print(np.abs(diff).max())


差の大きさの最大値は`5.68e-14`、つまり0.0000000000000568です。小数の計算の丸めで生じる、0と見なせる値です。2,412軒すべてで、居住面積は、1階・2階・低品質の面積の和とぴったり同じです。


### 問い：居住面積だけを1増やせるか？

係数は「ほかの列を同じにしたまま、その列を1増やすと、予測が何ドル変わるか」でした。では、1階・2階・低品質の面積を同じにしたまま、居住面積だけを1㎡増やすことはできるでしょうか。考えを下に書いてください。


（ここに考えを書く）


できません。居住面積は3つの列の和なので、3つを同じにしたまま居住面積だけを増やす家は、あり得ないのです。

式の中で言えば、居住面積に$b$ドル、1階・2階・低品質の面積にそれぞれ$c_1, c_2, c_3$ドルの係数が付いているとき、予測に効くのは$b + c_1$のような和だけです。$b$を100ドル減らして、3つの係数をそれぞれ100ドル増やしても、どの家の予測も変わりません。データからは、和は決まっても、内訳は決まらないのです。第4回の車庫の台数と面積は「ほとんど同じ」でしたが、こちらは足すとぴったり同じになるので、内訳はまったく決まりません。これを完全な多重共線性（multicollinearity）と呼びます。

地下の面積も、地下の仕上げ1・仕上げ2・未仕上げの和です。30本の列には、こうした組が入っています。居住面積の係数が小さく、揺れたのは、内訳の決まらない係数の1つを、たまたま取り出していたからです。


### お手本：30本の式の係数を見る

- 役割：2,412軒すべてで30本の列の式を作り、係数を表示する
- 入力：表`df`と、30本の列の名前`cols`
- 出力：列ごとの係数


In [ ]:
m30 = LinearRegression().fit(df[cols], df['価格'])
print(pd.Series(m30.coef_, index=cols).round(0))


ほかにも、読みにくい係数が並びます。キッチン数は−18,377ドルで、キッチンが1つ多い家は安い、と読めてしまいます。寝室数も−9,025ドルのマイナスです。係数の多くは、似た列と内訳を分け合っています。

列を増やすと、当たりは良くなりました。でも、お客さんに説明できる係数は、遠のいてしまいました。


---


## 2. 係数に罰をかけると？


列を増やすと、似た列どうしで係数の内訳が決まらなくなりました。内訳が決まらないなら、いくつもある内訳の候補の中から、1つを選ぶ決まりを足せばよいのではないでしょうか。


### 正則化：係数の大きさに罰をかける

普通の線形回帰は、誤差の二乗和$\sum_i (y_i - \hat{y}_i)^2$が最も小さくなる係数を選びます。内訳が決まらないときは、誤差の二乗和が同じ最小値になる係数が、いくつもあります。

そこで、誤差の二乗和に、係数の大きさへの罰を足した量を、最も小さくする係数を選びます。

$$
\sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} b_j^2
$$

$b_j$は$j$番目の列の係数、$p$は列の数（今日は30）、$\alpha$（アルファ）は罰の強さです。係数が大きいほど罰が大きくなるので、誤差の二乗和がほとんど変わらないなら、係数が小さいほうが選ばれます。内訳が決まらなかった係数も、1つに決まります。

このように、係数の大きさに罰をかけて係数を選ぶことを、正則化と呼びます。係数の二乗の和に罰をかけるこの方法を、リッジ回帰（Ridge regression）と呼びます。$\alpha = 0$なら罰が無く、普通の線形回帰と同じです。


### 罰の前に、目盛りをそろえる

罰は、どの列の係数にも同じ$\alpha$でかかります。ところが、係数の大きさは列の目盛りで変わります。居住面積をm²でなく平方フィートで測れば、1単位が小さくなるので、係数も小さくなります。目盛りしだいで罰の効き方が変わるのは困ります。

そこで、罰をかける前に、第3回で学んだ標準化で、どの列も平均0・標準偏差（standard deviation）1にそろえます。$x_{ij}$は$i$番目の家の$j$番目の列の値、$\bar{x}_j$と$s_j$はその列の平均と標準偏差です。

$$
z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}
$$

標準化に使う平均と標準偏差は、訓練の家だけで決めます。テストの家は、訓練の物差しで標準化します（第3回と同じ）。


### お手本：ドキュメントで`Ridge`を読む

`Ridge`は、初めて使う道具です。使う前に、ドキュメント（説明書き）を読みます。`help(道具の名前)`で、その道具のドキュメントが表示されます。

- 役割：`Ridge`のドキュメントを表示する
- 入力：道具`Ridge`
- 出力：ドキュメント（長いので、最初の部分と、Parametersの`alpha`の項だけ読む）


In [ ]:
help(Ridge)


最初の数行に、`Ridge`が最小にする式（目的関数）が書いてあります。

```
Minimizes the objective function::

||y - Xw||^2_2 + alpha * ||w||^2_2
```

`||y - Xw||^2_2`が誤差の二乗和、`||w||^2_2`が係数の二乗の和です（`w`が係数）。上で書いた式と同じです。

Parameters（引数）の節には、`alpha`の説明があります。

```
alpha : float or array-like of shape (n_targets,), default=1.0
    Constant that multiplies the L2 term, controlling regularization
    strength. `alpha` must be a non-negative float i.e. in `[0, inf)`.
```

「L2の項（係数の二乗の和）に掛ける定数で、正則化の強さを決める。0以上の数」です。既定値（default）は1.0です。ドキュメントは、1行目の「名前 : 型, 既定値」、その下の説明、の順に読みます。


### おさらい：標準化の関数

第3回で書いた、標準化の関数です。訓練で決めた平均`mean`と標準偏差`std`を引数で受け取ります。


In [ ]:
def standardize(X, mean, std):
    return (X - mean) / std


### お手本：モデルを引数にした交差検証の関数

1節の`cv_score`を、正則化に使えるように直します。違いは2つです。

1. 「式を作る」部品を、引数`model`で受け取る。`LinearRegression()`でも`Ridge(alpha=100)`でも渡せる
2. 毎回、訓練の平均と標準偏差で、訓練とテストの両方を標準化してから式を作る

係数は標準化した目盛りで出るので、標準偏差`std`で割って、元の1単位あたり（1㎡あたり、1部屋あたりなど）のドルに戻してためます。もう1つ、係数がちょうど0になった列の数（`0の数`）もためます。リッジでは0のままですが、3節で使います。

- 役割：5分割の交差検証で、渡されたモデルを、標準化した列で評価する
- 入力：列の名前`cols`、モデル`model`、係数を見る列`watch`
- 出力：5行の表（列はMAE・係数・0の数）


In [ ]:
def cv_model(cols, model, watch):
    rows = []
    for tr_i, te_i in kf.split(df):
        tr_s, te_s = df.iloc[tr_i], df.iloc[te_i]
        mean, std = tr_s[cols].mean(), tr_s[cols].std()  # 訓練で
        Z_tr = standardize(tr_s[cols], mean, std)
        Z_te = standardize(te_s[cols], mean, std)
        m_s = model.fit(Z_tr, tr_s['価格'])
        coef_s = pd.Series(m_s.coef_, index=cols) / std  # 1単位に
        rows.append({'MAE': mae(te_s['価格'], m_s.predict(Z_te)),
                     '係数': round(coef_s[watch]),
                     '0の数': (m_s.coef_ == 0).sum()})
    return pd.DataFrame(rows)

print(cv_model(cols, LinearRegression(), '居住面積'))


`LinearRegression()`を渡すと、MAEは1節の`cv_score`と同じ5つの数になります。第3回で見たとおり、標準化しても、普通の線形回帰の予測は変わらないからです。関数を直しても、当たりが前と同じになることを確かめました。

一方で、居住面積の係数は385〜398ドルで、1節（358〜462ドル）と違う値になりました。予測は同じなのに、係数だけが変わったのです。内訳が決まらないので、計算の手順（標準化したかどうか）しだいで、どの内訳が出てくるかが変わります。


### 変えてみる：リッジを渡す

- 役割：`Ridge(alpha=100)`を渡して、交差検証で評価する
- 入力：モデル`Ridge(alpha=100)`
- 出力：5回分のMAE・居住面積の係数


In [ ]:
ridge = cv_model(cols, Ridge(alpha=100), '居住面積')
print(ridge)
print(ridge['MAE'].mean())


MAEの平均は17,106ドルで、普通の線形回帰（17,188ドル）とほとんど同じか、少し良いくらいです。居住面積の係数は、1㎡あたり338〜350ドルで、回ごとの揺れが小さくなりました。罰をかけたので、内訳が1つに決まったのです。


### 変えてみる：1階と2階の係数も見る

- 役割：リッジで、1階の面積と2階の面積の係数を見る
- 入力：`watch`を`'1階の面積'`・`'2階の面積'`に変える
- 出力：5回分の係数


In [ ]:
print(cv_model(cols, Ridge(alpha=100), '1階の面積')['係数'].values)
print(cv_model(cols, Ridge(alpha=100), '2階の面積')['係数'].values)


1階の面積は288〜308ドル、2階の面積は205〜228ドルです。リッジは、居住面積の効果を、居住面積・1階・2階の3つの列に分け合わせました。

係数の二乗の和に罰をかけると、1つの列に大きな係数を集めるより、似た列に少しずつ分けたほうが罰が小さくなります（例えば、600を1列に置くと二乗は360,000ですが、300ずつ2列に分けると180,000です）。だから、リッジは似た列の係数を近づけ、分け合わせます。

揺れは小さくなりましたが、「居住面積1㎡あたり約340ドル」とお客さんに言うと、1階や2階の分を見落とします。説明には、まだ使いにくい係数です。


---


## 3. 係数をちょうど0にできる？


リッジは、似た列に係数を分け合わせました。でも、説明しやすいのは、似た列のうち1本に係数を集め、残りを0にした式です。第4回で手で外したように、外す列を式に選ばせることはできないでしょうか。


### ラッソ：係数の絶対値に罰をかける

罰を、係数の二乗の和から、係数の絶対値の和に変えます。

$$
\frac{1}{2n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \lvert b_j \rvert
$$

この方法を、ラッソ（Lasso）と呼びます。絶対値の罰は、係数が小さくなっても、0になるまで同じ強さで係数を引っぱります。そのため、効きの弱い列の係数は、ちょうど0になります。0になった列は、式から外れたのと同じです。


### お手本：ドキュメントで`Lasso`を読む

- 役割：`Lasso`のドキュメントを表示する
- 入力：道具`Lasso`
- 出力：ドキュメント（最初の部分を読む）


In [ ]:
help(Lasso)


最初の数行の目的関数を、リッジと比べてください。

```
The optimization objective for Lasso is::

    (1 / (2 * n_samples)) * ||y - Xw||^2_2 + alpha * ||w||_1
```

`||w||_1`が係数の絶対値の和です。そして、誤差の二乗和の前に`(1 / (2 * n_samples))`、つまり$\frac{1}{2n}$が付いています。リッジの式には付いていませんでした。

誤差の二乗和を$2n$（今日は約3,900）で割ってから罰と比べるので、ラッソの`alpha`は、リッジの`alpha`と同じ数でも、まったく違う強さになります。`alpha`という同じ名前でも、物差しが違うのです。名前から想像せず、ドキュメントの式で確かめます。


### お手本：ラッソを渡す

- 役割：`Lasso(alpha=300)`を`cv_model`に渡して評価する
- 入力：モデル`Lasso(alpha=300)`
- 出力：5回分のMAE・居住面積の係数・0になった列の数


In [ ]:
lasso = cv_model(cols, Lasso(alpha=300), '居住面積')
print(lasso)
print(lasso['MAE'].mean())


MAEの平均は17,148ドルで、リッジや普通の線形回帰とほとんど同じです。そのうえで、毎回3〜5本の列の係数が、ちょうど0になりました。

居住面積の係数は、1㎡あたり609〜629ドルです。第4回の10本の式（675〜705ドル）に近い大きさに戻りました。


### 変えてみる：どの列が0になった？

- 役割：2,412軒すべてで、標準化してラッソの式を作り、係数が0になった列を表示する
- 入力：表`df`と、列の名前`cols`
- 出力：係数が0になった列の名前

全部の家で式を作るときも、標準化してから当てます。物差しは、全部の家の平均と標準偏差です。`coef[coef == 0]`は、係数が0の列だけを取り出します（第2回で学んだ、条件で絞る書き方です）。


In [ ]:
mean_all, std_all = df[cols].mean(), df[cols].std()
Z_all = standardize(df[cols], mean_all, std_all)
lasso_all = Lasso(alpha=300).fit(Z_all, df['価格'])
coef = pd.Series(lasso_all.coef_, index=cols)
print(list(coef[coef == 0].index))


0になったのは、地下の未仕上げ・2階の面積・バスルーム数・洗面所の数の4本です。

2階の面積は、居住面積と1階の面積があれば、居住面積から1階を引けばほぼ分かります。地下の未仕上げも、地下の面積と2つの仕上げの面積から分かります。和でつながった組の中から、ラッソは1本を外したのです。

バスルーム数と洗面所の数は、第4回で「居住面積・部屋数と合わせて読む」とした列です。ラッソは、ほかの列が持っている情報と重なる列を外しました。バスルームが価格に関係ない、という意味ではありません。


---


## 4. 罰の強さは、どう選ぶ？


リッジもラッソも、`alpha`の値で効き方が変わります。ここまでは、リッジに100、ラッソに300を使いましたが、この値はどう決めればよいのでしょうか。


### お手本：リッジの`alpha`を交差検証で選ぶ

`alpha`を変えると、当たりが変わります。第4回で、式を比べるときの標準の物差しは交差検証でした。`alpha`の値ごとに交差検証のMAEを測って、いちばん小さい値を選びます。

- 役割：`alpha`を1〜1000に変えてリッジを交差検証で評価し、MAEの平均を表にする
- 入力：`alpha`の候補の並び
- 出力：`alpha`ごとのMAEの平均と、居住面積の係数の平均

テストの家で`alpha`を選ぶと、テストの家に合わせて選んだことになり、まだ見ていない家での当たりを甘く見積もります（第2回で見た自己採点の甘さと同じ）。訓練の中で交差検証して選びます。


In [ ]:
rows = []
for a in [1, 10, 30, 100, 300, 1000]:
    res = cv_model(cols, Ridge(alpha=a), '居住面積')
    rows.append({'alpha': a, 'MAE': res['MAE'].mean(),
                 '居住面積の係数': res['係数'].mean()})
print(pd.DataFrame(rows))


MAEは、`alpha`が100のとき17,106ドルで、いちばん小さくなります。`alpha`を大きくするほど、居住面積の係数は小さくなります。罰が強いほど、係数が0の方へ縮むからです。`alpha`が1000まで強すぎると、MAEも17,547ドルと悪くなります。


In [ ]:
res_df = pd.DataFrame(rows)
plt.plot(res_df['alpha'], res_df['MAE'], 'o-')
plt.xscale('log')                           # 横軸を対数の目盛りに
plt.xlabel('alpha')
plt.ylabel('交差検証のMAE（ドル）')
plt.show()


`alpha`は1から1000まで、何倍も違う値を並べたので、横軸を対数の目盛り（`plt.xscale('log')`）にしています。MAEは、`alpha`が100のところで谷になります。罰が弱すぎても強すぎても、当たりは悪くなるのです。


### TODO②：ラッソの`alpha`も選ぶ

リッジのセルを写し、`Ridge(alpha=a)`を`Lasso(alpha=a)`に変えて、`alpha`を10・30・100・300・1000・3000にしたときの交差検証のMAEの平均を表にしてください。0になった列の数の平均（`res['0の数'].mean()`）も、表に加えてください。


In [ ]:
# TODO: alpha を 10〜3000 に変えてラッソを評価し、表にする

# 解答例②：ラッソの alpha を交差検証で選ぶ
rows = []
for a in [10, 30, 100, 300, 1000, 3000]:
    res = cv_model(cols, Lasso(alpha=a), '居住面積')
    rows.append({'alpha': a, 'MAE': res['MAE'].mean(),
                 '居住面積の係数': res['係数'].mean(),
                 '0の数': res['0の数'].mean()})
print(pd.DataFrame(rows))


ラッソでは、`alpha`が300のとき、MAEは17,148ドルでいちばん小さくなります。0になる列は、`alpha`が10や30では1本ですが、300で平均4本、1000で平均9.8本に増えます。3000まで強めると、平均15.6本が0になり、MAEも17,717ドルと悪くなります。

もう1つ、セルの出力に、赤い枠で`ConvergenceWarning`という警告が出たはずです。`alpha`が10や30のときの居住面積の係数（233ドル、467ドル）も、ほかの`alpha`と比べて不自然です。


### お手本：警告を読み、ドキュメントで`max_iter`を読む

警告の文には、こう書いてあります。

```
ConvergenceWarning: Objective did not converge. You might want to
increase the number of iterations, ...
```

「目的関数の最小値に、たどり着かなかった（収束しなかった）。くり返しの回数を増やすとよいかもしれない」です。ラッソは、係数を少しずつ動かす計算をくり返して、目的関数の最小値を探します。`alpha`が小さく罰が弱いと、たどり着くまでに多くのくり返しが必要になります。

`help(Lasso)`のParametersの節で、くり返しの回数を決める引数を探すと、`max_iter`があります。

```
max_iter : int, default=1000
    The maximum number of iterations.
```

既定値は1000回です。これを大きくして、`alpha`=10で確かめます。

- 役割：`max_iter`を10万回にして、`alpha`=10のラッソを交差検証で評価する
- 入力：モデル`Lasso(alpha=10, max_iter=100000)`
- 出力：5回分のMAE・居住面積の係数・0の数


In [ ]:
print(cv_model(cols, Lasso(alpha=10, max_iter=100000), '居住面積'))


警告は出なくなり、居住面積の係数は627〜669ドルになりました。収束（convergence）していない式の係数は、途中の値で、信じられません。警告は読み飛ばさず、ドキュメントで対処を確かめます。`alpha`が100以上では、既定の1000回で収束していました。


### お手本：`cross_val_score`のドキュメントを読む

交差検証は、scikit-learnにも道具があります。`cross_val_score`です。自分の関数と同じ数が出るか、確かめてみましょう。まず、ドキュメントで、引数`scoring`（何で評価するか）を読みます。


In [ ]:
help(cross_val_score)


Parametersの節に、`estimator`（モデル）・`X`・`y`・`scoring`・`cv`（分け方）などが並びます。`scoring`には評価の物差しの名前を文字で渡し、使える名前の一覧はウェブのドキュメント（User Guide）にあります。MAEの名前は`'neg_mean_absolute_error'`です。

`neg`はnegative（マイナス）の略です。User Guideには、scikit-learnの物差しは「値が大きいほど良い」にそろえてあり、MAEのように小さいほど良い物差しは、マイナスを付けて返す、と書いてあります。返ってきた値に、マイナスを付け直して使います。


### 変えてみる：自分の関数と比べる

- 役割：普通の線形回帰を、`cross_val_score`で、1節と同じ5つの組で評価する
- 入力：モデル`LinearRegression()`、30本の列、価格、分け方`kf`
- 出力：5回分のMAEと、その平均


In [ ]:
scores = cross_val_score(LinearRegression(), df[cols], df['価格'],
                         cv=kf, scoring='neg_mean_absolute_error')
print(scores)
print(-scores.mean())


返ってきた5つの値は、マイナスが付いています。マイナスを付け直した平均は17,188ドルほどで、1節の`cv_score`（17,188ドル）と同じです。自分で書いた関数と、ライブラリの関数が、同じ計算をしていることを確かめられました。

ただし、`cross_val_score`に`Ridge`を渡すときは注意が必要です。そのまま渡すと、標準化をせずに罰をかけてしまいます。各回の訓練で標準化してから罰をかけるには、自分の`cv_model`のように、標準化を交差検証の中に入れる必要があります。


---


## 5. 説明できる係数で査定する


罰の強さを選びました。リッジは`alpha`=100で17,106ドル、ラッソは`alpha`=300で17,148ドルと、当たりはほとんど同じです。説明に使うなら、似た列のうち1本に係数を集め、残りを0にするラッソの式のほうが読みやすいので、ラッソを使います。


### お手本：選んだ式の係数を、ドルで読む

- 役割：3節で全部の家で作ったラッソの式（`alpha`=300）の係数を、元の1単位あたりのドルに戻して表示する
- 入力：ラッソの式`lasso_all`と、全部の家の標準偏差`std_all`
- 出力：列ごとの係数（ドル）

標準化した目盛りの係数を、標準偏差で割ると、元の1単位あたりの係数に戻ります（`cv_model`の中でやったのと同じです）。


In [ ]:
coef_dollar = pd.Series(lasso_all.coef_, index=cols) / std_all
print(coef_dollar.round(0))


品質は1段階で13,953ドル、居住面積は1㎡で622ドル、車庫の面積は1㎡で266ドルです。居住面積は、第4回の10本の式（686ドル）に近い大きさで読めます。1階の面積（78ドル）は、居住面積に上乗せされる、1階にある部分の分です。

それでも、ラッソで全部の係数が読みやすくなったわけではありません。キッチン数は−17,708ドル、寝室数は−7,829ドルのままです。「ほかの列を同じにしたら」の読み方は、ここでも欠かせません。同じ居住面積で寝室が1つ多ければ、1部屋あたりが狭くなります。


### お手本：1軒目の家を査定する

- 役割：1軒目の家を、全部の家の物差しで標準化し、ラッソの式で査定する
- 入力：1軒目の家`df.iloc[[0]]`
- 出力：査定額

新しい家も、式を作ったときと同じ物差し（`mean_all`・`std_all`）で標準化してから当てます。


In [ ]:
house = df.iloc[[0]]
Z_h = standardize(house[cols], mean_all, std_all)
pred_h = lasso_all.predict(Z_h)[0]
price = house['価格'].iloc[0]
print(f'査定額：{pred_h:,.0f}ドル（実際の価格は{price:,}ドル）')


1軒目の家の査定額は206,773ドルです。実際の価格は215,000ドルでした。第4回の10本の式では211,548ドルでした。

査定額は数千ドル違いますが、どちらが良いかは1軒では決められません。式の良し悪しは、交差検証で比べます。30本のラッソの式（17,148ドル）は、第4回の10本の式（19,297ドル）より、平均で2千ドルほど外れが小さいのです。


### TODO③：別の家を査定する

1軒目の代わりに、`df.iloc[[5]]`（6軒目の家）を、ラッソの式で査定してください。お手本のセルを写し、`0`を`5`に変えるだけで動きます。


In [ ]:
# TODO: 6軒目の家（df.iloc[[5]]）をラッソの式で査定する

# 解答例③：6軒目の家を査定する
house = df.iloc[[5]]
Z_h = standardize(house[cols], mean_all, std_all)
pred_h = lasso_all.predict(Z_h)[0]
price = house['価格'].iloc[0]
print(f'査定額：{pred_h:,.0f}ドル（実際の価格は{price:,}ドル）')


6軒目の家の査定額は196,095ドルで、実際の価格は195,500ドルでした。


### 答え

列を30本に増やすと、交差検証のMAEは19,297ドルから17,188ドルに小さくなりました。ただ、居住面積が1階・2階・低品質の面積の和であるように、足すとほかの列になる列があり、係数の内訳は決まりませんでした。

係数の絶対値に罰をかけるラッソ（`alpha`=300、交差検証で選んだ）を使うと、当たりは17,148ドルのまま、和でつながった組や重なる列のうち4本の係数が0になり、居住面積は1㎡あたり622ドルと、説明に使える大きさに戻りました。当たりを良くしながら、説明できる係数を保つことはできます。ただし、「ほかの列を同じにしたら」を付けて読む必要は残ります。


まだ言えないこと

- 外す列は揺れないか：ラッソが0にする列は、分け方を変えると入れ替わることがある（発展で見る）
- 使わなかった列：間口の長さは欠測の理由が分からず使わなかった。第3回の回帰補完で戻せば、手がかりになるかもしれない
- 文字の列：地区や外壁の材料などの文字の列は、まだ式に入れていない
- 列が家の数より多いとき：今日は30本と2,412軒だった。列のほうがずっと多いデータでは、普通の式はどうなるか（発展で見る）


---


## 目標に答えられたか


- 今日の目標は「列を増やして当たりを良くしながら、お客さんに説明できる係数を保てるか」でした。答えと、その理由を説明できますか？
- 1節で、居住面積の係数が10本の式と違ったのはなぜですか？ 居住面積だけを1増やすことはできますか？
- 2節で、罰をかける前に標準化したのはなぜですか？ 標準化に使う平均と標準偏差は、どの家で決めましたか？
- 2節のリッジと3節のラッソで、居住面積の係数はどう違いましたか？ なぜ違うのでしょうか？
- 3節で、ラッソの`alpha`とリッジの`alpha`は、なぜ同じ数でも強さが違うのですか？ どこを読んで分かりましたか？
- 4節で、`alpha`をテストの家でなく交差検証で選んだのはなぜですか？
- 4節で、`cross_val_score`の値にマイナスが付いていたのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：列を増やすと当たりは良くなるが、似た列の係数は決まらなくなる。係数に罰をかけて（正則化）係数を落ち着かせ、罰の強さは交差検証で選ぶ
- 足すとほかの列になる列があると（完全な多重共線性）、係数の和は決まっても内訳は決まらない
- 正則化は、誤差の二乗和に係数の大きさへの罰を足して、係数を1つに決める。罰の前に、訓練の物差しで標準化する
- リッジ（二乗の罰）は係数を縮めて似た列で分け合い、ラッソ（絶対値の罰）はいくつかの係数をちょうど0にする
- 罰の強さ`alpha`は、交差検証で選ぶ。弱すぎても強すぎても当たりは悪くなる
- Python：初めての道具は、ドキュメントで目的関数と引数を読んでから使う。同じ名前の引数でも、物差しが違うことがある。モデルを関数の引数にして、交差検証の部品を差し替える


---


## 次回へ


ここまでの5回は、価格や台数のような数を当ててきました。でも、当てたいものが数ではなく、「病気か、そうでないか」のような区分だったら、どうすればよいでしょうか。

次回の問いは、検査の値から、患者が心臓病かどうかを、どう見分けるか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

4節で、ラッソの`alpha`を強めると、0になる列が増えました。`alpha`を1000にしたラッソで、`cv_model`による交差検証をしてください。そのうえで、2,412軒すべてで（3節の`Z_all`を使って）`alpha`=1000のラッソの式を作り、係数が0になった列の名前と、その数を表示してください。


In [ ]:
lasso1000 = cv_model(cols, Lasso(alpha=1000), '居住面積')
print(lasso1000)
print(lasso1000['MAE'].mean())
l1000 = Lasso(alpha=1000).fit(Z_all, df['価格'])
coef1000 = pd.Series(l1000.coef_, index=cols)
print(list(coef1000[coef1000 == 0].index))
print((coef1000 == 0).sum(), '本')


<details><summary>詰まったら</summary>

交差検証は`cv_model(cols, Lasso(alpha=1000), '居住面積')`です。MAEの平均を、`alpha`=300（17,148ドル）と比べましょう。
全部の家の式は、3節のセルの`Lasso(alpha=300)`を`Lasso(alpha=1000)`に変えます。0になった列の数は`len(…)`か`(coef == 0).sum()`で数えられます。

</details>


### 応用②（判断）

応用①で、2,412軒すべてで作った`alpha`=1000のラッソの式では、係数が0になった列は何本ですか。整数で答えてください（例: 3）。


In [ ]:
print('答え:', (coef1000 == 0).sum())


<details><summary>詰まったら</summary>

`alpha`=300のときは4本でした。

</details>


### 応用③（解釈）

応用①の結果をもとに、お客さんへの説明に使う式として、`alpha`=300と`alpha`=1000のどちらのラッソを選ぶかを、当たりの面と説明のしやすさの面から、3行程度で書いてください。


（模範例）

`alpha`=1000のラッソは、交差検証のMAEが17,193ドルで、`alpha`=300（17,148ドル）とほとんど変わりません。そのうえで、0になる列が4本から9本に増え、部屋数やポーチ・プールの面積も式から外れます。

お客さんへの説明では、効く列が少ないほうが話しやすいので、`alpha`=1000の式を選びます。ただし、外れた列が価格に関係ないわけではなく、ほかの列と重なっているだけだと、付け加えて説明します。


---


## 発展（任意）


### ラッソが0にする列は、分け方で変わるか

3節で、2,412軒すべてで作ったラッソの式では、4本の列の係数が0になりました。交差検証の5回の訓練それぞれで式を作ったら、同じ列が0になるでしょうか。

- 役割：5回の訓練それぞれで、標準化してラッソ（`alpha`=300）の式を作り、0になった列を表示する
- 入力：分け方`kf`、30本の列`cols`
- 出力：回ごとの、0になった列の名前


In [ ]:
for tr_i, te_i in kf.split(df):
    tr_s = df.iloc[tr_i]
    mean, std = tr_s[cols].mean(), tr_s[cols].std()
    m_s = Lasso(alpha=300).fit(standardize(tr_s[cols], mean, std),
                               tr_s['価格'])
    coef_s = pd.Series(m_s.coef_, index=cols)
    print(list(coef_s[coef_s == 0].index))


5回とも0になるのは、地下の未仕上げ・2階の面積・洗面所の数の3本です。和でつながった組の1本は、どの分け方でも外れます。一方、バスルーム数が0になるのは2回だけで、開放ポーチ・囲いポーチ・プールの面積も、1回ずつ0になります。

効きの弱い列や、ほかの列と重なる列は、訓練に入る家が少し変わるだけで、残ったり外れたりします。ラッソが外した列を「価格に関係ない列」と読むのは危険です。どの列を外すかも、第4回で見た係数の揺れと同じく、データの1回分の偶然を含んでいます。


### 列が家の数より多いとき：遺伝子発現データ

今日のデータは、2,412軒に対して30本の列でした。列のほうが行よりずっと多いデータでは、何が起きるでしょうか。正則化がいちばん力を発揮するのは、こうしたデータです。

ここで使うのは、ラットの目の組織で、遺伝子がどれくらい働いているか（発現量）を測ったデータです（Scheetzほか, 2006, PNAS。NCBI GEOのGSE5680）。120匹のラットについて、TRIM32という遺伝子の発現量を、ほかの200個の遺伝子の発現量から当てます。TRIM32は、目の病気（バルデー・ビードル症候群）の原因遺伝子の1つです。値は、発現量の対数（底は2）です。200個の遺伝子は、元の約3万1千個の測定から、120匹での分散が大きいものを選びました。

1行が1匹、1列が1つの遺伝子です。行（120）より列（200）のほうが多いのです。


In [ ]:
rat = pd.read_csv(f'{DATA_DIR}/rat_eye.csv', index_col=0)
y_rat = rat['TRIM32']
X_rat = rat.drop(columns='TRIM32')
print(X_rat.shape)
print(round(np.abs(y_rat - y_rat.mean()).mean(), 3))  # 平均で答える


120匹 × 200遺伝子です。すべてのラットに平均の値を答えると、MAEは0.094です。

交差検証の各回の訓練は96匹です。96匹に対して200本の列があるので、普通の線形回帰は、訓練の96匹を誤差0で通る式をいくらでも作れます（第4回で見た「内訳が決まらない」が、もっと極端になります）。

- 役割：普通の線形回帰とラッソを、標準化して5分割の交差検証で評価し、訓練でのR²とテストでのMAEを比べる
- 入力：`X_rat`・`y_rat`、モデル
- 出力：モデルごとの、訓練でのR²の平均・テストでのMAEの平均・0でない係数の数


In [ ]:
def cv_rat(model):
    r2, err, used = [], [], []
    for tr_i, te_i in kf.split(X_rat):
        X_tr, X_te = X_rat.iloc[tr_i], X_rat.iloc[te_i]
        y_tr, y_te = y_rat.iloc[tr_i], y_rat.iloc[te_i]
        mean, std = X_tr.mean(), X_tr.std()
        m_s = model.fit(standardize(X_tr, mean, std), y_tr)
        r2.append(m_s.score(standardize(X_tr, mean, std), y_tr))
        pred_s = m_s.predict(standardize(X_te, mean, std))
        err.append(np.abs(y_te - pred_s).mean())
        used.append((m_s.coef_ != 0).sum())
    print(round(np.mean(r2), 3), round(np.mean(err), 3), used)

cv_rat(LinearRegression())
cv_rat(Lasso(alpha=0.02, max_iter=100000))


普通の線形回帰は、訓練でのR²が1.0です。訓練のラットを完璧に当てています。ところが、テストでのMAEは0.122で、平均で答えたとき（0.094）より悪くなります。列が多すぎて、訓練のラットに合わせすぎたのです。

ラッソ（`alpha`=0.02）は、訓練でのR²は0.5ほどに下がりますが、テストでのMAEは0.087で、平均で答えるより良くなります。使った遺伝子は、200個のうち、回ごとに9〜18個です。

列が行より多いデータでは、正則化なしでは当たる式が作れません。遺伝子のデータで、正則化（とくにラッソ）が広く使われるようになったのは、このためです。ここでは`max_iter`（くり返しの上限）を、既定値の1000から大きくしています。ドキュメントで`max_iter`の説明を読み、なぜ大きくするとよいのかも確かめてみましょう。


### `RidgeCV`と`LassoCV`：交差検証で`alpha`を選ぶ道具

4節では、`alpha`を`for`文で変えて、自分の`cv_model`で選びました。scikit-learnには、交差検証で`alpha`まで選んでくれる`RidgeCV`と`LassoCV`があります。ドキュメントで、引数`alphas`と`cv`を読んでから使ってみましょう。

ただし、`cross_val_score`と同じく、そのまま渡すと標準化を交差検証の中で行いません。下のセルは、全部の家で標準化した`Z_all`を使っているので、テストの家の情報が物差しに混ざっています。4節の自分の`for`の結果と比べて、選ばれる`alpha`が近いかを確かめるだけにします。


In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV
r_cv = RidgeCV(alphas=[1, 10, 30, 100, 300, 1000], cv=kf,
               scoring='neg_mean_absolute_error')
r_cv.fit(Z_all, df['価格'])
l_cv = LassoCV(alphas=[10, 30, 100, 300, 1000, 3000], cv=kf,
               max_iter=100000).fit(Z_all, df['価格'])
print(r_cv.alpha_, l_cv.alpha_)


リッジは100で、4節の自分の`for`と同じ`alpha`が選ばれました。ラッソは30で、4節の300と違います。`LassoCV`のドキュメントのAttributes（式を作ったあとに付く値）の節には`mse_path_`（各回のテストでの誤差の二乗の平均）があり、MAEではなく誤差の二乗の平均で`alpha`を選んでいることが分かります。何で選ぶかが違えば、選ばれる値も変わります。

試すなら、`LassoCV`のドキュメントで`alphas`を読みましょう。リストの代わりに整数を渡すと、その個数の`alpha`を自動で並べて試します。`alphas=100`にして使ってみましょう。
